In [3]:
# run_analysis_conversational_final.py

import os
import json
import re
from datetime import datetime
from typing import List, Dict, Optional

# 确保您已安装必要的库: pip install langchain langchain-openai pydantic
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

from dotenv import load_dotenv

# --- Pydantic模型定义 ---
class ScoreComponent(BaseModel):
    reason: str = Field(description="The reason for assigning these points.")
    points: int = Field(description="The points for this specific reason (can be positive or negative).")

class ScoreCalculation(BaseModel):
    components: List[ScoreComponent]
    final_snippet_score: int

class Evidence(BaseModel):
    snippet: str
    score_calculation: ScoreCalculation
    reasoning_summary: str

class AnalysisResult(BaseModel):
    holistic_analysis: str
    evidence_list: List[Evidence]

# --- 辅助函数 ---
def load_risk_events(filepath: str = "./Data/LLM/risk_events_8.json") -> List[Dict]:
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"错误：加载风险事件文件 '{filepath}' 失败: {e}")
        return []

def load_prompt_template(filepath: str = "./Data/LLM/prompt_template_v6.txt") -> str:
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()
    except Exception as e:
        print(f"错误：加载Prompt模板文件 '{filepath}' 失败: {e}")
        return ""

def parse_filename(filepath: str) -> Dict[str, Optional[str]]:
    try:
        basename = os.path.basename(filepath)
        match = re.match(r"([A-Z]+)_(\d{4}-\d{2}-\d{2})\.txt", basename)
        if match:
            return {"company_name": match.group(1), "filing_date": match.group(2)}
    except Exception:
        pass
    return {"company_name": "Unknown", "filing_date": "Unknown"}

# --- 主分析函数 ---
def run_full_analysis(document_filepath: str):
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        print("错误：OPENAI_API_KEY 未设置。")
        return

    # --- 加载配置 ---
    custom_risk_events = load_risk_events()
    # 加载完整的Prompt模板字符串
    full_prompt_template_string = load_prompt_template()
    file_info = parse_filename(document_filepath)
    
    if not custom_risk_events or not full_prompt_template_string:
        print("由于配置文件缺失，无法继续。")
        return

    # --- 初始化对话链 ---
    llm = ChatOpenAI(model="o3", temperature=1, model_kwargs={"response_format": {"type": "json_object"}})
    # 使用一个简单的模板，因为我们将手动构建完整的输入
    memory = ConversationBufferMemory()
    conversation = ConversationChain(llm=llm, memory=memory, verbose=False)

    try:
        with open(document_filepath, "r", encoding="utf-8") as f:
            document_content = f.read()
    except FileNotFoundError:
        print(f"错误：在 '{document_filepath}' 未找到文档文件。")
        return

    # --- 循环处理每个事件 ---
    all_final_reports = []
    print(f"--- 开始为文档 '{document_filepath}' 进行连续对话分析 ---")
    
    for i, event in enumerate(custom_risk_events):
        print(f"\n({i+1}/{len(custom_risk_events)}) 正在分析事件: {event['event_name']}...")
        
        # 动态构建Prompt
        if i == 0:
            # 第一次调用，使用完整的Prompt模板
            prompt_engine = PromptTemplate(
                template=full_prompt_template_string,
                input_variables=["document_text", "company_name", "filing_date", "event_name", "event_timeframe", "event_description"],
                partial_variables={"format_instructions": JsonOutputParser(pydantic_object=AnalysisResult).get_format_instructions()}
            )
            current_input = prompt_engine.format(
                document_text=document_content,
                company_name=file_info["company_name"],
                filing_date=file_info["filing_date"],
                **event
            )
        else:
            # 后续调用，使用简洁的指令
            current_input = build_followup_prompt(event, file_info)
            
        try:
            # 调用对话链并解析结果
            response_text = conversation.predict(input=current_input)
            analysis_data = json.loads(response_text)
            
            # 计算分数
            evidence_list = analysis_data.get('evidence_list', [])
            total_score = sum(e.get('score_calculation', {}).get('final_snippet_score', 0) for e in evidence_list)
            final_score = max(0, min(100, total_score))

            all_final_reports.append({
                "event_info": event,
                "holistic_analysis": analysis_data['holistic_analysis'],
                "evidence_list": evidence_list,
                "calculated_score": final_score
            })
            print(f"  > 分析完成. 找到 {len(evidence_list)} 条证据. 计算得分为: {final_score}")

        except Exception as e:
            print(f"  > 在分析此事件时发生错误: {e}")
    
    print_summary_report(all_final_reports, file_info)

def build_followup_prompt(event: Dict, file_info: Dict) -> str:
    """构建后续对话的简洁Prompt。"""
    return f"""
Excellent. Now, using the same document I provided initially, please perform the exact same two-step analysis (Holistic Analysis -> Evidence Micro-Scoring) for the following new event. Adhere to all principles and formatting from our first interaction.

**CONTEXT FOR ANALYSIS:**
* **Company:** {file_info['company_name']}
* **Filing Date:** {file_info['filing_date']}

**EVENT TO ANALYZE:**
* **Event Name:** {event['event_name']}
* **Event Timeframe:** {event['event_timeframe']}
* **Analysis Focus:** {event['event_description']}

Produce your output in the same valid JSON format as before.
"""

def print_summary_report(reports: List[Dict], file_info: Dict):
    """用于打印详细摘要报告的辅助函数。"""
    if not reports:
        print("\n无分析结果可显示。")
        return

    print("\n\n======================================================")
    print(f"  为 {file_info['company_name']} ({file_info['filing_date']}) 生成的连续对话分析报告")
    print("======================================================")
    
    sorted_reports = sorted(reports, key=lambda x: x.get('calculated_score', 0), reverse=True)

    for report in sorted_reports:
        event = report['event_info']
        score = report['calculated_score']
        holistic_analysis = report['holistic_analysis']
        evidence_list = report['evidence_list']

        print(f"\n--- 事件: {event.get('event_name', 'N/A')} ---")
        print(f"  最终计算得分 (0-100): {score}")
        print(f"\n  整体分析 (思维链):")
        print(f"    {holistic_analysis}")
        
        if evidence_list:
            sorted_evidence = sorted(evidence_list, key=lambda x: x.get('score_calculation', {}).get('final_snippet_score', 0), reverse=True)
            print("\n  证据分解 (按严重性排序):")
            for ev in sorted_evidence:
                calc = ev.get('score_calculation', {})
                snippet_score = calc.get('final_snippet_score', 0)
                print(f"    [总分: {snippet_score}/10] \"{ev.get('snippet', '')}\"")
                print(f"      摘要: {ev.get('reasoning_summary', '')}")
                print(f"      记分卡:")
                for comp in calc.get('components', []):
                    sign = "+" if comp.get('points', 0) >= 0 else ""
                    print(f"        [{sign}{comp.get('points')}] {comp.get('reason')}")
        else:
            print("\n  证据分解: 未找到相关证据。")
            
    print("\n======================================================")


if __name__ == "__main__":
    run_full_analysis(document_filepath='AAPL_2023-11-03.txt')  # 替换为实际的文档路径


--- 开始为文档 'AAPL_2023-11-03.txt' 进行连续对话分析 ---

(1/8) 正在分析事件: Global_Financial_Crisis_Subprime...
  > 分析完成. 找到 7 条证据. 计算得分为: 19

(2/8) 正在分析事件: Arab_Spring...
  > 分析完成. 找到 4 条证据. 计算得分为: 9

(3/8) 正在分析事件: Paris_Agreement_on_Climate_Change...
  > 分析完成. 找到 5 条证据. 计算得分为: 24

(4/8) 正在分析事件: Brexit_Referendum...
  > 分析完成. 找到 5 条证据. 计算得分为: 22

(5/8) 正在分析事件: COVID_19_Pandemic...
  > 分析完成. 找到 4 条证据. 计算得分为: 13

(6/8) 正在分析事件: Russia_Ukraine_Conflict...
  > 分析完成. 找到 4 条证据. 计算得分为: 15

(7/8) 正在分析事件: US_China_Chip_Export_Controls...
  > 分析完成. 找到 8 条证据. 计算得分为: 35

(8/8) 正在分析事件: Israel_Hamas_Conflict...
  > 分析完成. 找到 5 条证据. 计算得分为: 22


  为 AAPL (2023-11-03) 生成的连续对话分析报告

--- 事件: US_China_Chip_Export_Controls ---
  最终计算得分 (0-100): 35

  整体分析 (思维链):
    Apple’s 2023 Form 10-K does not cite the October 2022 U.S.-China semiconductor export-control regime by name, yet it repeatedly warns that new or tightened “controls on imports or exports of goods, technology or data” could force costly supplier changes, redesig